In [2]:
# Importing libraries and setup
from dotenv import load_dotenv
import requests
from agents import Runner, trace, Agent, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)

MODEL_NAME = 'gpt-4o-mini'

In [3]:
# Configurations
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

In [4]:
# Function to send the email
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = EMAIL_ADDRESS
    msg['Subject'] = subject 
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype='html')

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [5]:
# Testing the function
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

### Push notification setup

In [6]:
# Configuration
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# function to send push notification
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [7]:
USE_EMAIL = True # if you want to send notification instead of email

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [8]:
# Testing
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

## Agent Orchestration

There are 2 models for Agent Orchestration  
by code and by LLMs.  
**By code**: more predictable and deterministic.  
**By LLMs**: more powerful.  

An excellent write-up is here:  
https://openai.github.io/openai-agents-python/multi_agent/

### Orchestration by code

In [13]:
intro = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility"
instructions2 = intro + "Your email style is witty, engagging, and humorous"
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [14]:
print(instructions1)


You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
Your email style is professional, serious, with gravitas and credibility


In [15]:
# Creating different sales agents
sales_agent1 = Agent(name='Professional Sales Agent', instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)


In [22]:
from IPython.display import display, Markdown
from IPython.display import clear_output
from openai.types.responses import ResponseTextDeltaEvent

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")

response = ""
async for event in result.stream_events():

    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        response += event.data.delta
        
        clear_output(wait=True)
        display(Markdown(response))

Subject: Elevate Your Compliance Standards with AI-Driven Solutions

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a leader in AI-powered solutions designed to streamline and fortify SOC 2 compliance.

As you know, ensuring compliance is not just a regulatory requirement but a critical component for building trust with clients. The complexities of maintaining SOC 2 standards can be daunting, often requiring extensive resources and meticulous attention to detail.

ComplAI offers an innovative SaaS tool that simplifies the compliance process. Our platform provides:

- **Real-time Monitoring:** Stay ahead of compliance requirements with automated updates and alerts.
- **Comprehensive Documentation:** Effortlessly generate and manage the necessary documentation for audits.
- **Risk Assessment:** Utilize AI to identify potential vulnerabilities before they become issues.

By integrating ComplAI into your compliance strategy, you can significantly reduce the time and effort needed to prepare for audits, allowing your team to focus on what they do best.

I would welcome the opportunity to discuss how ComplAI can support your organization's compliance goals. Please let me know a convenient time for you, or feel free to schedule a brief call via [insert scheduling link].

Thank you for your time. I look forward to the possibility of working together.

Best regards,

[Your Name]  
[Your Position]  
ComplAI  
[Your Phone Number]  
[Your Email Address]  
[Your Company Website]  

In [23]:
# Running all three agents and storing their results

message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in results]

for output in outputs:
    display(Markdown(output))
    print(f"\n{'--'* 50}\n")

Subject: Elevate Your Compliance Strategy with Advanced AI Solutions

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a leading provider of SaaS solutions designed to streamline SOC2 compliance and audit preparedness.

In an increasingly complex regulatory environment, achieving and maintaining compliance can be a daunting challenge. At ComplAI, we leverage cutting-edge artificial intelligence technology to simplify the process, enabling organizations to efficiently navigate compliance requirements while minimizing risks.

Our platform offers:

- **Automated Documentation**: Reduce the manual workload with smart tools that generate and manage essential compliance documents.
- **Real-Time Monitoring**: Gain continuous insights into your compliance status, allowing for proactive adjustments and remediation.
- **Comprehensive Reporting**: Simplify audit preparations with concise, structured reports that meet SOC2 standards.

Successful organizations recognize that robust compliance is not merely a regulatory obligation; it is a cornerstone of trust and credibility with your clients.

I would welcome the opportunity to discuss how ComplAI can support your compliance initiatives and help position your organization for sustained success. Would you be available for a brief call next week? 

Thank you for considering this opportunity. I look forward to the possibility of working together.

Best regards,

[Your Name]  
[Your Title]  
ComplAI  
[Your Phone Number]  
[Your Email Address]  
[Company Website]  


----------------------------------------------------------------------------------------------------



Subject: 🚀 Ready to Get SOC2 Compliant Without Losing Your Mind?

Hi [Recipient's Name],

Are you tired of juggling compliance like it’s a flaming sword? You’re not alone! 🤹‍♂️ 

Imagine this: instead of losing sleep over SOC2 audits, you could be binge-watching your favorite series—or, you know, actually getting work done. Enter ComplAI, your new best friend in the world of compliance.

Our AI-powered tool does all the heavy lifting for you—think of it as your compliance superhero, minus the cape (caped superheroes have serious wardrobe malfunctions). With ComplAI on your side, preparing for audits will feel less like a circus act and more like, well, a smooth ride.

Curious to learn more? Let’s set up a quick chat where I promise not to juggle anything, and you can finally kick compliance chaos to the curb.

Looking forward to hearing from you!

Best,

[Your Name]  
[Your Job Title]  
ComplAI  
[Your Email]  
[Your Phone Number]  

P.S. Don’t worry—we won’t take your sanity in exchange for compliance! 😉


----------------------------------------------------------------------------------------------------



Subject: Streamline Your SOC 2 Compliance with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. I’m reaching out to introduce ComplAI, an AI-powered SaaS tool designed to simplify SOC 2 compliance and streamline your audit process.

Our solution automates key compliance tasks, reduces manual effort, and ensures you meet the necessary requirements efficiently. Companies like yours have seen significant time savings and reduced headaches during audits.

I'd love to schedule a brief call to discuss how ComplAI can enhance your compliance strategy.

Best,  
[Your Name]  
[Your Job Title]  
ComplAI  
[Your Phone Number]  
[Your LinkedIn Profile]  


----------------------------------------------------------------------------------------------------



In [25]:
# sales picker
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only
"""

sales_picker = Agent(name='sales_picker', instructions=decision, model=MODEL_NAME)

In [26]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

In [28]:
display(Markdown(best.final_output))

Subject: Don’t Let SOC 2 Compliance Be the Unicorn You Can’t Catch 🦄

Hey [Recipient's Name],

I hope this email finds you somewhere between conquering your to-do list and sipping your favorite coffee (or tea, no judgment here!). 

I wanted to drop you a quick note about something we both know can be as fun as watching paint dry: SOC 2 compliance. Let’s be honest—audits can feel like a never-ending game of hide and seek, and the paperwork can pile up faster than a toddler’s Lego collection!

Enter ComplAI: the hero in your compliance saga! Our AI-powered SaaS tool is designed to take the dread out of SOC 2 compliance (and maybe even turn it into a net positive). Imagine a world where audits feel like a walk in the park (with a piñata and snacks, of course)!

With ComplAI, you'll be able to:
- **Automate the boring stuff**: Say goodbye to manual tasks that make you want to pull your hair out.
- **Stay organized**: (Because who couldn’t use a little more organization in their life?)
- **Prepare like a pro**: Walk into audits with confidence, ready to dazzle the auditors with your impeccable documentation.

Would you be open to a quick chat? I promise it’ll be more fun than finding an old pizza slice in the office fridge (but if that’s your thing, I won’t judge!).

Looking forward to hearing from you!

Best,  
[Your Name]  
[Your Position]  
ComplAI  
[Your Phone Number]  
[Your LinkedIn Profile or Website]  

P.S. Just think of us as the “please don’t leave me hanging” in the world of compliance! 🎉

### Adding a tool to the mix

In [30]:
@function_tool 
def send_email_tool(subject:str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """

    send_email(subject, text_body, html_body)
    return "Email Sent Successfully"

In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

required_tools = ModelSettings(tool_choice="required")

sales_sender = Agent(name='Sales Sender', instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=required_tools)

In [34]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)
    print(f"Final response:\n{best.final_output}")

Final response:
Subject: Don’t Let SOC 2 Compliance Be the Unicorn You Can’t Catch 🦄

Hey [Recipient's Name],

I hope this email finds you somewhere between conquering your to-do list and sipping your favorite coffee (or tea, no judgment here!). 

I wanted to drop you a quick note about something we both know can be as fun as watching paint dry: SOC 2 compliance. Let’s be honest—audits can feel like a never-ending game of hide and seek, and the paperwork can pile up faster than a toddler’s Lego collection!

Enter ComplAI: the hero in your compliance saga! Our AI-powered SaaS tool is designed to take the dread out of SOC 2 compliance (and maybe even turn it into a net positive). Imagine a world where audits feel like a walk in the park (with a piñata and snacks, of course)!

With ComplAI, you'll be able to:
- **Automate the boring stuff**: Say goodbye to manual tasks that make you want to pull your hair out.
- **Stay organized**: (Because who couldn’t use a little more organization in t

## Orchestrating by LLMs

#### A: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.  
The OpenAI Agents SDK gives a very simple way to do this.  

This works best when the flow is:  
Agent A -> Agent B -> Agent A  
And for the classic "Planning Agent" situation.

In [35]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name='sales_email_writer_1', tool_description=description)
tool1

FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000019010E3F800>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

#### So now we can gather all the tools togther:

A tool for each of our 3 email-writing agents  
And a tool for our function to send emails

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name='sales_email_writer_1', tool_description=description)
tool2 = sales_agent2.as_tool(tool_name='sales_email_writer_2', tool_description=description)
tool3 = sales_agent3.as_tool(tool_name='sales_email_writer_3', tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]
tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000019010E3D610>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input

In [40]:
# Now it's time for our sales manager -- our planning agent
instructions = """
You are a Sales Manager at ComplAI.

Your job is to create and send ONE cold sales email.

Follow this exact workflow:

1. Call sales_email_writer_1 exactly once.
2. Call sales_email_writer_2 exactly once.
3. Call sales_email_writer_3 exactly once.
4. Compare the three generated emails.
5. Select the single best email.
6. Call send_email_tool exactly once with ONLY the selected email.
7. After sending the email, stop. Do not call any other tools.

Do not call any sales writer more than once.
Do not send more than one email.
"""

task = """
Generate three different sales email drafts using the three sales email
writer tools.

Then select the best draft and send ONLY that draft using send_email_tool.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)

In [41]:
with trace("Sales Manager"):
    result = await Runner.run(sales_manager, task)